# YUMEHO 見込み施設抽出ツール

ハローワーク等から収集した求人データをもとに、  
YUMEHO の導入ターゲットとなる施設を **自動スコアリング** するツールです。

---

## 使い方

1. **Step 1** のセルを実行（初回のみ・準備）
2. **Step 2-A**（ハローワーク自動検索）または **Step 2-B**（手動CSVアップロード）を実行
3. **Step 3** でスコアリング結果を確認
4. **Step 4** で Excel をダウンロード

---

### スコアリング基準

| 条件 | 加点 |
|------|------|
| 施設種別（回復期リハビリ病院、老健、デイケア等） | +20 |
| 職種（PT/OT） | +15 |
| 職種（介護職） | +10 |
| 歩行リハビリ直接言及 | +25 |
| 転倒関連言及 | +15 |
| 急募 | +15 |
| 複数名募集（2名〜） | +10〜50 |
| 人手不足・欠員補充等のシグナル | 各+5 |
| 同一施設が複数職種で求人 | +10〜20 |

### 優先度

| ランク | スコア | アクション |
|--------|--------|----------|
| **A（最優先）** | 60点以上 | 即アプローチ |
| **B（有望）** | 35点以上 | 優先的にアプローチ |
| **C（標準）** | 15点以上 | リスト保持 |
| **D（低）** | 15点未満 | 見送り |

---
## Step 1: 準備（初回のみ実行）

In [ ]:
# ── 必要なライブラリをインストール ──
!pip install -q openpyxl beautifulsoup4 requests

import csv
import io
import json
import os
import re
import time
from collections import defaultdict
from datetime import datetime
from urllib.parse import urljoin

import requests
from bs4 import BeautifulSoup
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side

try:
    from google.colab import files as colab_files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

import pandas as pd
from IPython.display import display, HTML

print('\u2713 準備完了')

In [ ]:
# ══════════════════════════════════════════════════════
#  スコアリングエンジン（このセルは自動実行されます）
# ══════════════════════════════════════════════════════

TARGET_FACILITY_TYPES = [
    "回復期リハビリテーション", "回復期リハビリ", "介護老人保健施設", "老健",
    "通所リハビリ", "デイケア", "デイサービス", "リハビリテーション病院",
    "リハビリ病院", "地域包括ケア", "訪問リハビリ", "介護医療院",
    "特別養護老人ホーム", "特養", "有料老人ホーム",
]

HIGH_VALUE_SIGNALS = [
    "人手不足", "急募", "増員", "欠員補充", "即日", "随時",
    "未経験可", "資格不問", "経験不問", "ブランク可",
    "複数名", "2名", "3名",
    "歩行介助", "歩行訓練", "転倒", "転倒防止", "身体介助", "移乗",
]

AREA_CODES = {
    "北海道": "01", "青森県": "02", "岩手県": "03", "宮城県": "04",
    "秋田県": "05", "山形県": "06", "福島県": "07", "茨城県": "08",
    "栃木県": "09", "群馬県": "10", "埼玉県": "11", "千葉県": "12",
    "東京都": "13", "神奈川県": "14", "新潟県": "15", "富山県": "16",
    "石川県": "17", "福井県": "18", "山梨県": "19", "長野県": "20",
    "岐阜県": "21", "静岡県": "22", "愛知県": "23", "三重県": "24",
    "滋賀県": "25", "京都府": "26", "大阪府": "27", "兵庫県": "28",
    "奈良県": "29", "和歌山県": "30", "鳥取県": "31", "島根県": "32",
    "岡山県": "33", "広島県": "34", "山口県": "35", "徳島県": "36",
    "香川県": "37", "愛媛県": "38", "高知県": "39", "福岡県": "40",
    "佐賀県": "41", "長崎県": "42", "熊本県": "43", "大分県": "44",
    "宮崎県": "45", "鹿児島県": "46", "沖縄県": "47",
}


def score_listing(listing):
    """求人情報をスコアリングする。"""
    score = 0
    reasons = []
    text = " ".join([
        str(listing.get("job_title", "")),
        str(listing.get("company_name", "")),
        str(listing.get("description", "")),
        str(listing.get("requirements", "")),
    ]).lower()

    for ft in TARGET_FACILITY_TYPES:
        if ft.lower() in text:
            score += 20
            reasons.append(f"施設種別: {ft}")
            break

    job = str(listing.get("job_title", ""))
    if re.search(r"理学療法士|PT", job):
        score += 15; reasons.append("PT求人")
    elif re.search(r"作業療法士|OT", job):
        score += 15; reasons.append("OT求人")
    elif re.search(r"介護職|介護員|ケアワーカー", job):
        score += 10; reasons.append("介護職求人")
    elif re.search(r"機能訓練|リハビリ", job):
        score += 12; reasons.append("リハビリ関連求人")

    for signal in HIGH_VALUE_SIGNALS:
        if signal in text:
            score += 5; reasons.append(f"シグナル: {signal}")

    if re.search(r"歩行.*リハビリ|歩行.*訓練|歩行.*介助", text):
        score += 25; reasons.append("歩行リハビリ直接言及")
    if "転倒" in text:
        score += 15; reasons.append("転倒関連言及")

    match = re.search(r"(\d+)名.*募集|募集.*(\d+)名", text)
    if match:
        num = int(match.group(1) or match.group(2))
        if num >= 2:
            score += 10 * min(num, 5)
            reasons.append(f"複数名募集: {num}名")

    if "急募" in text:
        score += 15; reasons.append("急募")

    listing["score"] = score
    listing["score_reasons"] = "; ".join(reasons[:8])
    listing["priority"] = (
        "A (最優先)" if score >= 60 else
        "B (有望)" if score >= 35 else
        "C (標準)" if score >= 15 else
        "D (低)"
    )
    return listing


def normalize_company_name(name):
    name = str(name).strip()
    name = re.sub(r"[\s\u3000]+", " ", name)
    name = re.sub(r"（医）|[(（]医療法人[)）]", "医療法人", name)
    name = re.sub(r"（社）|[(（]社会福祉法人[)）]", "社会福祉法人", name)
    name = re.sub(r"（福）", "社会福祉法人", name)
    return name


def deduplicate_and_aggregate(listings):
    """同一施設の複数求人を名寄せし、施設単位で集約する。"""
    facilities = defaultdict(lambda: {
        "company_name": "", "location": "", "job_count": 0,
        "job_titles": [], "best_score": 0, "all_reasons": set(),
        "priority": "D (低)", "descriptions": [],
        "job_numbers": [], "retrieved_at": "",
    })

    for listing in listings:
        name = normalize_company_name(listing.get("company_name", ""))
        if not name:
            continue
        f = facilities[name]
        f["company_name"] = listing.get("company_name", name)
        f["location"] = f["location"] or str(listing.get("location", ""))
        f["job_count"] += 1
        if listing.get("job_title"): f["job_titles"].append(str(listing["job_title"]))
        if listing.get("job_number"): f["job_numbers"].append(str(listing["job_number"]))
        if listing.get("description"): f["descriptions"].append(str(listing["description"])[:200])
        f["retrieved_at"] = str(listing.get("retrieved_at", ""))

        scored = score_listing(listing)
        if scored["score"] > f["best_score"]:
            f["best_score"] = scored["score"]
            f["priority"] = scored["priority"]
        if scored.get("score_reasons"):
            f["all_reasons"].update(scored["score_reasons"].split("; "))

    result = []
    for name, f in facilities.items():
        if f["job_count"] >= 3:
            f["best_score"] += 20
            f["all_reasons"].add(f"複数職種で求人: {f['job_count']}件")
        elif f["job_count"] >= 2:
            f["best_score"] += 10
            f["all_reasons"].add(f"複数求人: {f['job_count']}件")

        s = f["best_score"]
        f["priority"] = (
            "A (最優先)" if s >= 60 else
            "B (有望)" if s >= 35 else
            "C (標準)" if s >= 15 else
            "D (低)"
        )

        result.append({
            "優先度": f["priority"],
            "スコア": f["best_score"],
            "施設名": f["company_name"],
            "所在地": f["location"],
            "求人数": f["job_count"],
            "募集職種": " / ".join(list(dict.fromkeys(f["job_titles"]))[:5]),
            "判定理由": "; ".join(sorted(f["all_reasons"])[:8]),
            "求人番号": ", ".join(f["job_numbers"][:3]),
            "取得日": f["retrieved_at"],
            "概要": (f["descriptions"][0][:150] + "...") if f["descriptions"] else "",
            "アクション": "",
        })

    result.sort(key=lambda x: x["スコア"], reverse=True)
    return result


print('\u2713 スコアリングエンジン読み込み完了')

---
## Step 2-A: ハローワーク自動検索

以下のセルを実行すると、ハローワークインターネットサービスから  
求人情報を自動取得してスコアリングします。

**エリアを変更したい場合**は `search_areas` を編集してください。

In [ ]:
# ── 検索設定（必要に応じて変更してください） ──

search_areas = [
    "東京都",
    "神奈川県",
    "埼玉県",
    "千葉県",
    "大阪府",
    "愛知県",
]

search_keywords = [
    "理学療法士 回復期",
    "理学療法士 リハビリ",
    "作業療法士 リハビリ",
    "介護職員 リハビリ",
    "機能訓練指導員",
    "歩行訓練 介護",
]

max_pages_per_search = 3  # 各検索の最大ページ数
request_interval = 3      # リクエスト間隔（秒）

In [ ]:
# ── ハローワーク検索エンジン ──

BASE_URL = "https://www.hellowork.mhlw.go.jp"
SEARCH_URL = f"{BASE_URL}/kensaku/GECA110010.do"
HEADERS = {
    "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36",
    "Accept": "text/html,application/xhtml+xml",
    "Accept-Language": "ja,en-US;q=0.9,en;q=0.8",
}

def hw_search(keyword, area_name=None, max_pages=3):
    """ハローワークを検索して求人リストを返す。"""
    session = requests.Session()
    session.headers.update(HEADERS)
    results = []

    params = {
        "screenId": "GECA110010",
        "action": "searchAction",
        "searchTarget": "1",
        "freeWordType": "1",
        "freeWord": keyword,
        "searchButton": "検索",
    }
    if area_name and area_name in AREA_CODES:
        params["tDFK1CmbBox"] = AREA_CODES[area_name]

    try:
        resp = session.get(SEARCH_URL, params=params, timeout=30)
        resp.raise_for_status()

        for page in range(max_pages):
            soup = BeautifulSoup(resp.text, "html.parser")
            cards = soup.select(".kyujin-row, .job-item, .result-item, tr.job")

            if not cards:
                # パターンマッチでフォールバック
                body = soup.get_text()
                parts = re.split(r"(?=\d{5}-\d{8,})", body)
                for part in parts[1:]:
                    num_match = re.search(r"(\d{5}-\d{8,})", part)
                    listing = {
                        "job_number": num_match.group(1) if num_match else "",
                        "job_title": "",
                        "company_name": "",
                        "location": "",
                        "description": part[:500],
                        "requirements": "",
                        "source": "ハローワーク",
                        "retrieved_at": datetime.now().strftime("%Y-%m-%d"),
                    }
                    # テキストから施設名・所在地を推定
                    loc = re.search(
                        r"(北海道|青森|岩手|宮城|秋田|山形|福島|茨城|栃木|群馬|"
                        r"埼玉|千葉|東京|神奈川|新潟|富山|石川|福井|山梨|長野|"
                        r"岐阜|静岡|愛知|三重|滋賀|京都|大阪|兵庫|奈良|和歌山|"
                        r"鳥取|島根|岡山|広島|山口|徳島|香川|愛媛|高知|福岡|"
                        r"佐賀|長崎|熊本|大分|宮崎|鹿児島|沖縄)[都道府県]?.{0,20}",
                        part
                    )
                    if loc: listing["location"] = loc.group(0)[:30]
                    results.append(listing)
                break

            for card in cards:
                text = card.get_text(separator=" ", strip=True)
                listing = {
                    "job_number": "", "job_title": "", "company_name": "",
                    "location": "", "description": text[:500],
                    "requirements": "", "source": "ハローワーク",
                    "retrieved_at": datetime.now().strftime("%Y-%m-%d"),
                }
                num_match = re.search(r"(\d{5}-\d{8,})", text)
                if num_match: listing["job_number"] = num_match.group(1)

                links = card.select("a")
                for link in links:
                    lt = link.get_text(strip=True)
                    if lt and len(lt) > 2:
                        if not listing["job_title"]: listing["job_title"] = lt
                        elif not listing["company_name"]: listing["company_name"] = lt

                for th in card.select("th, dt"):
                    label = th.get_text(strip=True)
                    td = th.find_next_sibling("td") or th.find_next_sibling("dd")
                    if not td: continue
                    value = td.get_text(strip=True)
                    if "事業所名" in label or "会社名" in label: listing["company_name"] = value
                    elif "職種" in label: listing["job_title"] = value
                    elif "就業場所" in label or "勤務地" in label: listing["location"] = value
                    elif "仕事の内容" in label: listing["description"] = value

                if listing.get("company_name"):
                    results.append(listing)

            # 次ページ
            next_link = None
            for a in soup.select("a"):
                if "次へ" in a.get_text() or "次の" in a.get_text():
                    next_link = a.get("href")
                    break
            if not next_link: break
            time.sleep(request_interval)
            resp = session.get(urljoin(BASE_URL, next_link), timeout=30)

    except Exception as e:
        print(f"    [エラー] {e}")

    return results

# ── 検索実行 ──
print("ハローワーク検索を開始します...")
print(f"  エリア: {', '.join(search_areas)}")
print(f"  キーワード: {len(search_keywords)} パターン")
print()

all_listings = []
for area in search_areas:
    print(f"── {area} ──")
    for kw in search_keywords:
        print(f"  検索: {kw}", end="")
        found = hw_search(kw, area_name=area, max_pages=max_pages_per_search)
        all_listings.extend(found)
        print(f" → {len(found)} 件")
        time.sleep(request_interval)

print(f"\n取得完了: 全 {len(all_listings)} 件")

# スコアリング
facilities = deduplicate_and_aggregate(all_listings)
print(f"名寄せ後: {len(facilities)} 施設")

---
## Step 2-B: 手動 CSV アップロード（自動検索の代替）

ハローワーク等から手動で収集したCSVをアップロードしてスコアリングできます。

### CSV フォーマット

以下のいずれかの列名を含む CSV をアップロードしてください:

| 列名 | 内容 | 必須 |
|------|------|------|
| `施設名` or `company_name` | 施設・法人名 | ○ |
| `職種` or `job_title` | 募集職種 | ○ |
| `所在地` or `location` | 所在地 | △ |
| `概要` or `description` | 求人内容 | △ |
| `求人番号` or `job_number` | 求人番号 | △ |

In [ ]:
# ── CSV アップロード ──
# このセルを実行するとファイル選択ダイアログが開きます

if IN_COLAB:
    print("CSV ファイルを選択してください...")
    uploaded = colab_files.upload()
    filename = list(uploaded.keys())[0]
    content = uploaded[filename].decode("utf-8-sig")
else:
    # ローカル実行時はパスを指定
    csv_path = input("CSV ファイルパスを入力: ")
    with open(csv_path, "r", encoding="utf-8-sig") as f:
        content = f.read()
    filename = os.path.basename(csv_path)

# CSV 読み込み
reader = csv.DictReader(io.StringIO(content))
all_listings = []
for row in reader:
    listing = {
        "job_title": row.get("職種", row.get("募集職種", row.get("job_title", ""))),
        "company_name": row.get("施設名", row.get("company_name", "")),
        "location": row.get("所在地", row.get("location", "")),
        "description": row.get("概要", row.get("description", row.get("仕事の内容", ""))),
        "requirements": row.get("requirements", row.get("必要な経験等", "")),
        "job_number": row.get("求人番号", row.get("job_number", "")),
        "retrieved_at": row.get("取得日", row.get("retrieved_at", datetime.now().strftime("%Y-%m-%d"))),
    }
    all_listings.append(listing)

print(f"\n読み込み完了: {len(all_listings)} 件 ({filename})")

# スコアリング
facilities = deduplicate_and_aggregate(all_listings)
print(f"名寄せ後: {len(facilities)} 施設")

---
## Step 3: スコアリング結果を確認

In [ ]:
# ── サマリー表示 ──

total = len(facilities)
by_priority = defaultdict(int)
for f in facilities:
    by_priority[f["優先度"]] += 1

summary_html = f"""
<div style="font-family: 'Yu Gothic', sans-serif; padding: 20px; background: #f8f9fa; border-radius: 12px; margin: 10px 0;">
  <h2 style="color: #0068B7; margin-top: 0;">YUMEHO 見込み施設リスト</h2>
  <p style="font-size: 1.1em;">合計 <strong>{total}</strong> 施設</p>
  <div style="display: flex; gap: 16px; flex-wrap: wrap;">
    <div style="background: #FFF2CC; padding: 16px 24px; border-radius: 8px; min-width: 120px; text-align: center;">
      <div style="font-size: 2em; font-weight: bold; color: #B7791F;">{by_priority.get('A (最優先)', 0)}</div>
      <div style="font-size: 0.85em; color: #744210;">A（最優先）</div>
    </div>
    <div style="background: #E8F4FD; padding: 16px 24px; border-radius: 8px; min-width: 120px; text-align: center;">
      <div style="font-size: 2em; font-weight: bold; color: #0068B7;">{by_priority.get('B (有望)', 0)}</div>
      <div style="font-size: 0.85em; color: #2C5282;">B（有望）</div>
    </div>
    <div style="background: #F0F0F0; padding: 16px 24px; border-radius: 8px; min-width: 120px; text-align: center;">
      <div style="font-size: 2em; font-weight: bold; color: #555;">{by_priority.get('C (標準)', 0)}</div>
      <div style="font-size: 0.85em; color: #666;">C（標準）</div>
    </div>
    <div style="background: #fff; padding: 16px 24px; border-radius: 8px; min-width: 120px; text-align: center; border: 1px solid #ddd;">
      <div style="font-size: 2em; font-weight: bold; color: #999;">{by_priority.get('D (低)', 0)}</div>
      <div style="font-size: 0.85em; color: #999;">D（低）</div>
    </div>
  </div>
</div>
"""
display(HTML(summary_html))

In [ ]:
# ── 全施設リスト（テーブル表示） ──

df = pd.DataFrame(facilities)
display_cols = ["優先度", "スコア", "施設名", "所在地", "求人数", "募集職種", "判定理由"]
existing_cols = [c for c in display_cols if c in df.columns]

def highlight_priority(row):
    p = row.get("優先度", "")
    if "A" in str(p): return ["background-color: #FFF2CC"] * len(row)
    if "B" in str(p): return ["background-color: #E8F4FD"] * len(row)
    if "C" in str(p): return ["background-color: #F5F5F5"] * len(row)
    return [""] * len(row)

if not df.empty:
    styled = df[existing_cols].style.apply(highlight_priority, axis=1)
    display(styled)
else:
    print("結果がありません。Step 2 を先に実行してください。")

---
## Step 4: Excel ダウンロード

In [ ]:
# ── Excel 出力＆ダウンロード ──

def export_excel(facilities, filepath):
    wb = Workbook()
    ws = wb.active
    ws.title = "見込み施設リスト"

    header_fill = PatternFill(start_color="0068B7", end_color="0068B7", fill_type="solid")
    header_font = Font(name="Yu Gothic", bold=True, color="FFFFFF", size=10)
    cell_font = Font(name="Yu Gothic", size=10)
    thin_border = Border(
        left=Side(style="thin", color="D4DDE8"),
        right=Side(style="thin", color="D4DDE8"),
        top=Side(style="thin", color="D4DDE8"),
        bottom=Side(style="thin", color="D4DDE8"),
    )
    priority_fills = {
        "A (最優先)": PatternFill(start_color="FFF2CC", fill_type="solid"),
        "B (有望)": PatternFill(start_color="E8F4FD", fill_type="solid"),
        "C (標準)": PatternFill(start_color="F5F5F5", fill_type="solid"),
        "D (低)": PatternFill(start_color="FFFFFF", fill_type="solid"),
    }

    headers = ["優先度", "スコア", "施設名", "所在地", "求人数",
               "募集職種", "判定理由", "求人番号", "取得日", "アクション", "メモ"]
    col_widths = [12, 8, 30, 20, 8, 25, 35, 18, 12, 15, 20]

    for col, (header, width) in enumerate(zip(headers, col_widths), 1):
        cell = ws.cell(row=1, column=col, value=header)
        cell.fill = header_fill
        cell.font = header_font
        cell.alignment = Alignment(horizontal="center", vertical="center")
        cell.border = thin_border
        ws.column_dimensions[chr(64 + col) if col <= 26 else 'A' + chr(64 + col - 26)].width = width

    for row_idx, facility in enumerate(facilities, 2):
        priority = facility.get("優先度", "D (低)")
        fill = priority_fills.get(priority, priority_fills["D (低)"])

        values = [
            facility.get("優先度", ""), facility.get("スコア", 0),
            facility.get("施設名", ""), facility.get("所在地", ""),
            facility.get("求人数", 0), facility.get("募集職種", ""),
            facility.get("判定理由", ""), facility.get("求人番号", ""),
            facility.get("取得日", ""), "", "",
        ]
        for col, value in enumerate(values, 1):
            cell = ws.cell(row=row_idx, column=col, value=value)
            cell.font = cell_font
            cell.fill = fill
            cell.border = thin_border
            cell.alignment = Alignment(vertical="center", wrap_text=(col in [6, 7]))

    ws.auto_filter.ref = f"A1:K{len(facilities) + 1}"
    ws.freeze_panes = "A2"
    wb.save(filepath)


timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
xlsx_filename = f"yumeho_leads_{timestamp}.xlsx"
export_excel(facilities, xlsx_filename)

print(f"Excel 作成完了: {xlsx_filename}")
print(f"  A（最優先）: {by_priority.get('A (最優先)', 0)} 施設")
print(f"  B（有望）:   {by_priority.get('B (有望)', 0)} 施設")
print(f"  C（標準）:   {by_priority.get('C (標準)', 0)} 施設")
print(f"  合計:       {len(facilities)} 施設")

if IN_COLAB:
    colab_files.download(xlsx_filename)
    print("\nダウンロードが開始されます...")
else:
    print(f"\nファイル保存先: {os.path.abspath(xlsx_filename)}")

---
## 補足: 手動リサーチ用テンプレート CSV

以下のセルを実行すると、手動リサーチ用の空テンプレート CSV をダウンロードできます。  
このテンプレートに求人情報を入力し、Step 2-B でアップロードしてください。

In [ ]:
# ── テンプレート CSV ダウンロード ──

template = """施設名,職種,所在地,仕事の内容,必要な経験等,求人番号,取得日
医療法人○○会 ○○病院,理学療法士,東京都○○区,回復期リハビリ病棟での歩行訓練。急募。2名募集。,PT免許必須,13070-12345678,2026-04-12
社会福祉法人 ○○会,介護職員,神奈川県○○市,デイサービスでの介護業務。歩行介助・移乗介助。3名募集。未経験可。,初任者研修以上,14010-23456789,2026-04-12
"""

template_filename = "yumeho_leads_template.csv"
with open(template_filename, "w", encoding="utf-8-sig") as f:
    f.write(template)

if IN_COLAB:
    colab_files.download(template_filename)
    print("テンプレート CSV をダウンロードしました。")
else:
    print(f"テンプレート CSV: {os.path.abspath(template_filename)}")

print("\n記入例が2行入っています。削除して実データを入力してください。")
print("入力後、Step 2-B でアップロードするとスコアリングされます。")